# Block Ciphers

# AES

## 1. Introduction

The Advanced Encryption Standard (AES) is one of the most widely used symmetric encryption algorithms today. This report explores AES with a focus on its internal workings, particularly its confusion and diffusion properties. Using Monte Carlo simulations, we evaluate how these properties evolve across AES rounds.

This report also includes diagrams to clarify the transformation steps and structures used in AES, ensuring concepts are introduced clearly before they are used.

---

## 2. AES Overview

AES operates on a 128-bit block of plaintext, structured internally as a 4x4 matrix of bytes called the **State Matrix**. This matrix serves as the central data structure in AES transformations. Each transformation step manipulates the State Matrix in specific ways to ensure security.

### 2.1 State Matrix

The 128-bit input is interpreted as a 4x4 byte matrix:

```
| b00 b01 b02 b03 |
| b10 b11 b12 b13 |
| b20 b21 b22 b23 |
| b30 b31 b32 b33 |
```

Each column contains 4 bytes (32 bits). AES transformations operate on rows, columns, or the entire matrix.

---

## 3. AES Transformations

AES consists of multiple rounds of transformation steps:

1. **SubBytes** – Byte-wise substitution using a non-linear S-Box.
2. **ShiftRows** – Row-wise permutation.
3. **MixColumns** – Column-wise mixing using Galois Field arithmetic.
4. **AddRoundKey** – XOR operation with a round key derived from the main encryption key.

### 3.1 SubBytes

Each byte in the State Matrix is replaced using a substitution box (S-Box), providing non-linearity (confusion).

![SubBytes Diagram](https://upload.wikimedia.org/wikipedia/commons/a/a4/AES-SubBytes.svg)

### 3.2 ShiftRows

Rows in the State Matrix are cyclically shifted left by different offsets:

* Row 0: No shift
* Row 1: Shift by 1 byte
* Row 2: Shift by 2 bytes
* Row 3: Shift by 3 bytes

This introduces inter-column diffusion.

![ShiftRows Diagram](https://upload.wikimedia.org/wikipedia/commons/6/66/AES-ShiftRows.svg)

### 3.3 MixColumns

Each column is treated as a 4-byte polynomial and mixed via matrix multiplication over GF(2^8), spreading byte influence within the column.

![MixColumns Diagram](https://upload.wikimedia.org/wikipedia/commons/7/76/AES-MixColumns.svg)

### 3.3.1 Galois Field Arithmetic in AES

The MixColumns step of AES uses arithmetic in a finite field known as the Galois Field GF(2^8). In this field:

* Each byte (8 bits) is treated as a polynomial of degree ≤ 7 with coefficients in GF(2), i.e., modulo 2.
* Arithmetic operations (addition and multiplication) are defined over this field.

  * Addition: simple bitwise XOR
  * Multiplication: polynomial multiplication modulo an irreducible polynomial

AES specifically uses the irreducible polynomial:

x^8 + x^4 + x^3 + x + 1

This polynomial is represented in hexadecimal as 0x11b. It cannot be factored over GF(2), ensuring a valid field structure.

#### Why is this Important?

* During MixColumns, bytes are multiplied by constants (e.g., 0x01, 0x02, 0x03) using GF(2^8) multiplication.
* Without the irreducible polynomial, multiplication might yield results outside the field.
* For instance, multiplication by 0x02 involves left-shifting a byte and reducing modulo 0x11b when overflow occurs.

#### Example:

If b = 0x57 and we compute 0x02 · b in GF(2^8):

1. Left-shift 0x57 → 0xAE
2. Since there is no overflow, 0x02 · 0x57 = 0xAE

But if overflow occurs:

1. Left-shift 0xC0 → 0x180
2. XOR with 0x11B → 0x63 (to stay within 8 bits)

This operation ensures that every transformation remains within GF(2^8), preserving the field properties required by AES.

#### Matrix Multiplication in MixColumns

Each column of the state matrix is multiplied by a fixed matrix:

\| 02 03 01 01 |
\| 01 02 03 01 |
\| 01 01 02 03 |
\| 03 01 01 02 |

All multiplications are done in GF(2^8), using the irreducible polynomial above. This spreads byte influence within the column — a key mechanism for diffusion in AES.

### 3.4 AddRoundKey

Each byte of the State Matrix is XORed with the corresponding byte from the round key.

![AddRoundKey Diagram](https://upload.wikimedia.org/wikipedia/commons/a/ad/AES-AddRoundKey.svg)

### 3.5 AES Round Structure

AES-128 uses 10 rounds, with an initial AddRoundKey and a final round excluding MixColumns.

![AES Rounds Diagram](https://upload.wikimedia.org/wikipedia/commons/5/50/AES_%28Rijndael%29_Round_Function.png)

### 3.6 Key Expansion (Key Schedule)

AES uses a **Key Expansion** algorithm to generate a set of round keys from the original cipher key. These round keys are used in the AddRoundKey step during each round of encryption.

#### Why Key Expansion?

- AES-128 uses a 128-bit key but requires 11 round keys (one for each round plus an initial round key).
- The key expansion algorithm derives these keys from the original key to ensure security and avoid simple key reuse.
- Each round key is 128 bits (16 bytes), matching the block size.

#### Key Expansion Process Overview

- The original key is divided into 4 words (32 bits each).
- New words are generated iteratively using previous words, a **RotWord**, a **SubWord**, and round constants (Rcon).
- This process continues until enough words are generated for all rounds.
  

#### Steps in Key Expansion

1. **Start:** The original 128-bit key forms the first 4 words (W0 to W3).


2. **Iterate:** For each subsequent word \( W_i \):

- If \( i \) is a multiple of 4:
  - Rotate the previous word’s bytes left by one (RotWord).
  
  ![RotWord Step](https://davidwong.fr/blockbreakers/images/AES/key_schedule-07.jpg)  
  
  - Substitute each byte using the S-Box (SubWord).
  
  ![SubWord Step](https://braincoke.fr/assets/static/s-box.7a04a14.554cf219b54cc76564a39d839c8109eb.png)  
  *Figure: SubWord substitutes each byte using the AES S-Box*
  
  - XOR the result with the round constant \( Rcon_i \).

- Otherwise, simply XOR the previous word with the word 4 positions earlier.

---

#### Round Constants (Rcon)

Round constants add nonlinearity and ensure each round key differs.

![Rcon Diagram](https://captanu.wordpress.com/wp-content/uploads/2015/04/rcon.jpg)  

*Figure: Round constants used in key expansion*

---

#### Visual Summary

| Original Key (4 words)        | → Key Expansion →       | Round Keys (44 words)            |
|-------------------------------|-------------------------|---------------------------------|
| \( W_0, W_1, W_2, W_3 \)      |                         | \( W_0, W_1, ..., W_{43} \)     |

Each group of 4 words forms a round key for the corresponding AES round.

---

#### Testing the AES Module with the FIPS-197 Example:


In [ ]:
import numpy as np
from aes import AES

def print_state(state_bytes):
    state_matrix = np.frombuffer(state_bytes, dtype=np.uint8).reshape((4,4), order='F')
    for row in state_matrix:
        print(' '.join(f"{b:02x}" for b in row))
    print()

# Example key and plaintext
key = b'\x2b\x7e\x15\x16\x28\xae\xd2\xa6\xab\xf7\x15\x88\x09\xcf\x4f\x3c'
plaintext = b'\x32\x43\xf6\xa8\x88\x5a\x30\x8d\x31\x31\x98\xa2\xe0\x37\x07\x34'

# Instantiate AES
aes = AES(key)

# Expand keys
round_keys = aes.key_expansion(key)

# Initial AddRoundKey (round 0)
state = aes.add_round_key(plaintext, round_keys[0])
print("After Initial AddRoundKey (Round 0):")
print_state(state)

# Test Round 1 - each sub-step
print("Round 1 Steps:")

# Byte Substitution
state_bs = aes.byte_substitution(state)
print("After Byte Substitution:")
print_state(state_bs)

# Shift Rows
state_sr = aes.shift_rows(state_bs)
print("After Shift Rows:")
print_state(state_sr)

# Mix Columns
state_mc = aes.mix_column(state_sr)
print("After Mix Columns:")
print_state(state_mc)

# Add Round Key
state_ark = aes.add_round_key(state_mc, round_keys[1])
print("After Add Round Key (Round 1):")
print_state(state_ark)

# You can continue similarly for other rounds:
# For example, Round 2:
print("Round 2 Steps:")
state_bs = aes.byte_substitution(state_ark)
print("After Byte Substitution:")
print_state(state_bs)

state_sr = aes.shift_rows(state_bs)
print("After Shift Rows:")
print_state(state_sr)

state_mc = aes.mix_column(state_sr)
print("After Mix Columns:")
print_state(state_mc)

state_ark = aes.add_round_key(state_mc, round_keys[2])
print("After Add Round Key (Round 2):")
print_state(state_ark)

# ... and so on, up to round 9.

# For final round (round 10), skip Mix Columns and use final_round method if desired:
print("Final Round (Round 10) Steps:")
state_bs = aes.byte_substitution(state_ark)
print("After Byte Substitution:")
print_state(state_bs)

state_sr = aes.shift_rows(state_bs)
print("After Shift Rows:")
print_state(state_sr)

# No Mix Columns in final round
state_ark = aes.add_round_key(state_sr, round_keys[10])
print("After Add Round Key (Round 10):")
print_state(state_ark)

## 4. Confusion and Diffusion

* **Confusion** obscures the relationship between the key and ciphertext (achieved mainly via SubBytes).
* **Diffusion** spreads plaintext structure across ciphertext (achieved via ShiftRows and MixColumns).

These properties are essential to protect against statistical attacks.

---

## 5. Monte Carlo Simulation Methodology

To study the evolution of confusion and diffusion:

1. Perform `N = 300` trials.
2. In each trial:

   * Encrypt a random plaintext using AES for a given number of rounds.
   * Compute Hamming Distance:

     * For diffusion: between ciphertexts of two plaintexts differing by one bit.
     * For confusion: between ciphertexts encrypted with keys differing by one bit.
3. Collect and average the Hamming distances for each round.

This simulates the avalanche effect and how it grows with each round.

### 5.1 AES Monte Carlo Simulation Code and plot



In [ ]:
from collections import Counter
from diffusion_confusion import aes_diffusion, aes_confusion
import matplotlib.pyplot as plt

In [ ]:
from diffusion_confusion import aes_diffusion, aes_confusion

# Parameters
trials = 300
num_rounds = 11  # AES-128: 10 rounds + initial round

# Containers (lists for rounds 0 to 10)
results_diffusion = [[] for _ in range(num_rounds)]
results_confusion = [[] for _ in range(num_rounds)]

# Monte Carlo simulation
for _ in range(trials):
    for r in range(num_rounds):
        results_diffusion[r].append(aes_diffusion(num_round=r))
        results_confusion[r].append(aes_confusion(num_round=r))

# Calculate averages
avg_diffusion = [sum(vals) / len(vals) for vals in results_diffusion]
avg_confusion = [sum(vals) / len(vals) for vals in results_confusion]

# Calculate distributions
dist_diffusion = [Counter(vals) for vals in results_diffusion]
dist_confusion = [Counter(vals) for vals in results_confusion]

# Access:
# avg_diffusion[5] => average Hamming distance at round 5
# dist_confusion[10][64] => how many times distance 64 appeared in round 10 confusion


In [ ]:
# Define rounds to plot (1 to 10)
rounds_to_plot = range(1, 11)  # Skip round 0

# --- Plot Diffusion ---
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
fig.suptitle("AES Diffusion — Hamming Distance Distribution per Round", fontsize=16)

for ax, r in zip(axes.flat, rounds_to_plot):
    ax.bar(dist_diffusion[r].keys(), dist_diffusion[r].values(), color='skyblue')
    ax.set_title(f"Round {r} (Avg: {avg_diffusion[r]:.1f})")
    ax.set_xlim(0, 130)
    ax.set_xlabel("Hamming Distance")
    ax.set_ylabel("Frequency")
    ax.grid(True)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# --- Plot Confusion ---
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
fig.suptitle("AES Confusion — Hamming Distance Distribution per Round", fontsize=16)

for ax, r in zip(axes.flat, rounds_to_plot):
    ax.bar(dist_confusion[r].keys(), dist_confusion[r].values(), color='salmon')
    ax.set_title(f"Round {r} (Avg: {avg_confusion[r]:.1f})")
    ax.set_xlim(0, 130)
    ax.set_xlabel("Hamming Distance")
    ax.set_ylabel("Frequency")
    ax.grid(True)

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


### 💡Bonus Task: AES Modes of Operation

In this part of the project, we extend our AES-128 implementation by applying it within various **modes of operation**, which define how AES encrypts data longer than one 16-byte block.

While AES itself operates on 128-bit blocks, real-world data is typically longer. Modes of operation solve this by securely chaining blocks, each with specific rules. The most common modes we evaluate are:

- **ECB (Electronic Codebook):** The simplest mode. Each 16-byte block is encrypted independently. However, this mode is vulnerable to pattern leaks and is generally discouraged.
  
- **CBC (Cipher Block Chaining):** Each plaintext block is XORed with the previous ciphertext block before encryption. This ensures that identical plaintext blocks produce different ciphertexts (when using a unique IV).

- **CFB (Cipher Feedback Mode):** Converts AES into a stream cipher. It uses the output of AES encryption to produce a keystream, which is XORed with plaintext.

- **OFB (Output Feedback Mode):** Similar to CFB, but uses AES output in a feedback loop without involving the plaintext or ciphertext. Produces consistent keystream blocks.

- **CTR (Counter Mode):** AES is used to encrypt a counter value that changes with each block. This output is XORed with the plaintext to produce ciphertext.

### Objective

We will encrypt and decrypt a fixed plaintext using the same AES key under **all five modes**, observing how:
- The ciphertext changes between modes,
- The plaintext is accurately recovered,
- And how IVs and counters influence the encryption process.

This evaluation helps demonstrate the practical applications and differences of AES beyond its core block algorithm.

In [ ]:
from aes import AES
from blockcipher import BlockCipher

key = b'ThisIsA16ByteKey'
plaintext = b'choqos bezan azizam bodo bebinam'

# --- Encrypt ---
cipher_enc = BlockCipher(key=key, mode='CBC')

ciphertext = cipher_enc.encrypt(plaintext)

# Save IV from encryption
iv_used = cipher_enc.iv

# --- Decrypt ---
cipher_dec = BlockCipher(key=key, iv=iv_used, mode='CBC')
decrypted = cipher_dec.decrypt(ciphertext)
print("Original:   ", plaintext)
print("Ciphertext: ", ciphertext.hex())
print("Decrypted : ", decrypted)




In [ ]:
key = b'ThisIsA16ByteKey'
plaintext = b"AESMODETEST_AESMODETEST_AESMODETEST_"  # 36 bytes (multiple AES blocks)

modes = ['ECB', 'CBC', 'CFB', 'OFB']

print("=== AES Mode of Operation Testing ===")
for mode in modes:
    print(f"\n--- Testing Mode: {mode} ---")

    cipher_enc = BlockCipher(key=key, mode=mode)
    ciphertext = cipher_enc.encrypt(plaintext)

    iv_or_nonce = cipher_enc.iv if hasattr(cipher_enc, 'iv') else None
    if iv_or_nonce:
        print(f"Used IV/Nonce: {iv_or_nonce.hex()}")

    cipher_dec = BlockCipher(key=key, mode=mode, iv=iv_or_nonce)
    decrypted = cipher_dec.decrypt(ciphertext)

    print("Plaintext   :", plaintext)
    print("Ciphertext  :", ciphertext.hex())
    print("Decrypted   :", decrypted)
    print("Match       :", plaintext == decrypted)



### ✅ AES Modes Test – Results Summary

We tested **AES-128** encryption using the following four standard modes: **ECB**, **CBC**, **CFB**, and **OFB**. For all modes:

- The **original plaintext** was:  
  `b'AESMODETEST_AESMODETEST_AESMODETEST_'`  
  (a repeating, 32-byte string — ideal for observing block-level behavior)

- **Encryption and decryption succeeded** — all modes returned the exact original plaintext after decryption.

- Ciphertexts differed across modes, demonstrating their **unique operational mechanisms** and **security characteristics**.

---

### 🔍 Observations

| Mode | IV Used? | Pattern Breaking | Output Match | Suitable For             |
|------|----------|------------------|--------------|--------------------------|
| ECB  | ❌ No     | ❌ Weak           | ✅ Yes        | ❌ Not recommended        |
| CBC  | ✅ Yes    | ✅ Strong         | ✅ Yes        | ✅ Secure data storage    |
| CFB  | ✅ Yes    | ✅ Strong         | ✅ Yes        | ✅ Real-time data streams |
| OFB  | ✅ Yes    | ✅ Strong         | ✅ Yes        | ✅ Error-prone channels   |

> All modes maintained full decryption accuracy. However, **ECB reused identical ciphertext blocks** for repeated plaintext, confirming its vulnerability. In contrast, **CBC, CFB, and OFB** demonstrated strong diffusion and better randomness due to IV and chaining logic.


## 8. Conclusion

In this project, we implemented and analyzed **AES-128 encryption** from the ground up. We validated each core transformation step—AddRoundKey, SubBytes, ShiftRows, MixColumns, KeyExpansion—and confirmed correctness by comparing with the full `encrypt()` method.

We then evaluated **AES’s cryptographic strength** by analyzing:
- **Diffusion**: how a single bit change in plaintext affects the ciphertext.
- **Confusion**: how a single bit change in the key impacts the ciphertext.

Results showed Diffusion improves with more rounds. At 10 rounds, Hamming distance converges toward half the block size (\~64 bits), showing strong diffusion, and Confusion follows a similar pattern. It improves with rounds, ensuring that slight key differences cause significant ciphertext variation.



Finally, we extended our implementation to support and test four AES **modes of operation**: ECB, CBC, CFB, and OFB. We verified their encryption-decryption accuracy and highlighted the differences in ciphertext behavior and security.

Overall, this project demonstrates a comprehensive understanding of AES, both in theoretical foundations and practical application.

AES demonstrates robust confusion and diffusion characteristics, crucial for secure encryption. This report visualized how these properties build up over rounds and validated the design via simulation.

Monte Carlo analysis confirmed the avalanche effect: changing 1 bit in the plaintext or key results in \~50% bit changes in the output after sufficient rounds, confirming AES's strength against differential and linear cryptanalysis.

---

## 9. References

* FIPS PUB 197: Advanced Encryption Standard (AES)
* [https://en.wikipedia.org/wiki/Advanced\_Encryption\_Standard](https://en.wikipedia.org/wiki/Advanced_Encryption_Standard)
* Daemen, J., & Rijmen, V. (2002). The Design of Rijndael: AES — The Advanced Encryption Standard